You trade against a secret number of counterparties that all have a **reserve price** ranging between **670** and **920**. You trade at most once with each counterparty. On the next trading day, you’re able to sell all the product for a fair price, **920**.

The distribution of the bids is **uniformly distributed** at **increments of 5** between **670** and **920** (inclusive on both ends). 

**Example**: counterparties may have reserve prices at 675 and 680, but not at 676, 677, 678, 679, etc..

You may submit **two bids**. If the first bid is **higher** than the reserve price, they trade with you at your first bid. If your second bid is **higher** than the reserve price of a counterparty and **higher** than the mean of second bids of all players you trade at your second bid. If your second bid is **higher** than the reserve price, but **lower** than or **equal** to the mean of second bids of all players, the chance of a trade rapidly decreases: you will trade at your second bid **but** your PNL is penalised by

((920-avg_b2)/(920-b2))^3

In [ ]:
import numpy as np
import math



def PnL(bid1:int, bid2:int):
    reserve_price = 
    if bid1 > reserve_price
    

In [1]:
"""
Celestial Gardeners' Guild — manual trading challenge simulator.

Mechanics
---------
- Reserve prices are drawn uniformly from {670, 675, ..., 920} (51 values).
- Fair sell price next day = 920.
- You submit two bids (b1, b2). Conventionally b1 <= b2.
- For each counterparty with reserve r:
    * If b1 >= r            -> trade at b1, pnl = 920 - b1
    * Elif b2 >= r and b2 > avg_b2  -> trade at b2, pnl = 920 - b2
    * Elif b2 >= r and b2 <= avg_b2 -> trade at b2 but penalised
                                       pnl = (920 - b2) - ((920 - avg_b2)/(920 - b2))**3
                                       (penalty model is swappable; see PenaltyMode)
    * Else                  -> no trade, pnl = 0

Notes
-----
- The problem statement's penalty wording is ambiguous: "PNL is penalised by ..."
  could be subtracted from PnL (implemented here as the default), or used as a
  trade probability. Both modes are supported.
- avg_b2 is the mean of all *players'* second bids — in a real game you don't
  know it. In simulation we either set it exogenously (to study best response)
  or compute it from the population we simulate.
"""

from __future__ import annotations

from dataclasses import dataclass
from enum import Enum
from typing import Callable, Iterable

import numpy as np

FAIR_PRICE = 920
RESERVE_MIN = 670
RESERVE_MAX = 920
RESERVE_STEP = 5
RESERVES = np.arange(RESERVE_MIN, RESERVE_MAX + 1, RESERVE_STEP)  # 51 values


# ---------------------------------------------------------------------------
# Core: single-counterparty PnL given (b1, b2, reserve, avg_b2)
# ---------------------------------------------------------------------------

class PenaltyMode(str, Enum):
    SUBTRACT = "subtract"      # pnl = (920 - b2) - penalty_term
    PROBABILITY = "probability"  # trade with prob (1 - penalty_term), else 0


def _penalty_term(b2: float, avg_b2: float) -> float:
    """((920 - avg_b2) / (920 - b2)) ** 3, clipped to [0, 1] for stability."""
    if b2 >= FAIR_PRICE:
        return 1.0  # degenerate: no edge anyway
    raw = ((FAIR_PRICE - avg_b2) / (FAIR_PRICE - b2)) ** 3
    return float(np.clip(raw, 0.0, 1.0))


def trade_pnl(
    b1: float,
    b2: float,
    reserve: float,
    avg_b2: float,
    penalty_mode: PenaltyMode = PenaltyMode.SUBTRACT,
    rng: np.random.Generator | None = None,
) -> tuple[float, str]:
    """Return (pnl, outcome_label) for one counterparty."""
    if b1 >= reserve:
        return FAIR_PRICE - b1, "b1_hit"

    if b2 >= reserve:
        if b2 > avg_b2:
            return FAIR_PRICE - b2, "b2_clean"
        # b2 hit but penalised
        pen = _penalty_term(b2, avg_b2)
        if penalty_mode is PenaltyMode.SUBTRACT:
            return (FAIR_PRICE - b2) - pen, "b2_penalised"
        # PROBABILITY mode: trade happens with prob (1 - pen)
        rng = rng or np.random.default_rng()
        if rng.random() < (1.0 - pen):
            return FAIR_PRICE - b2, "b2_prob_traded"
        return 0.0, "b2_prob_skipped"

    return 0.0, "no_trade"


# ---------------------------------------------------------------------------
# Expected PnL per counterparty, vectorised over the reserve distribution
# ---------------------------------------------------------------------------

def expected_pnl_per_cp(
    b1: float,
    b2: float,
    avg_b2: float,
    penalty_mode: PenaltyMode = PenaltyMode.SUBTRACT,
) -> float:
    """Expectation over the uniform reserve distribution {670, 675, ..., 920}."""
    pnl_sum = 0.0
    for r in RESERVES:
        # In probability mode, take expectation over the Bernoulli too
        if b1 >= r:
            pnl_sum += FAIR_PRICE - b1
        elif b2 >= r:
            if b2 > avg_b2:
                pnl_sum += FAIR_PRICE - b2
            else:
                pen = _penalty_term(b2, avg_b2)
                if penalty_mode is PenaltyMode.SUBTRACT:
                    pnl_sum += (FAIR_PRICE - b2) - pen
                else:  # PROBABILITY
                    pnl_sum += (1.0 - pen) * (FAIR_PRICE - b2)
        # else no trade, contributes 0
    return pnl_sum / len(RESERVES)


# ---------------------------------------------------------------------------
# Monte-Carlo simulation over many counterparties
# ---------------------------------------------------------------------------

@dataclass
class SimResult:
    b1: float
    b2: float
    avg_b2: float
    n_counterparties: int
    total_pnl: float
    mean_pnl: float
    outcome_counts: dict[str, int]

    def __str__(self) -> str:
        return (
            f"b1={self.b1:.1f}  b2={self.b2:.1f}  avg_b2={self.avg_b2:.1f}  "
            f"N={self.n_counterparties}\n"
            f"  total PnL = {self.total_pnl:.2f}   mean PnL/cp = {self.mean_pnl:.4f}\n"
            f"  outcomes  = {self.outcome_counts}"
        )


def simulate(
    b1: float,
    b2: float,
    avg_b2: float,
    n_counterparties: int = 100_000,
    penalty_mode: PenaltyMode = PenaltyMode.SUBTRACT,
    seed: int | None = 0,
) -> SimResult:
    rng = np.random.default_rng(seed)
    reserves = rng.choice(RESERVES, size=n_counterparties)

    counts = {"b1_hit": 0, "b2_clean": 0, "b2_penalised": 0,
              "b2_prob_traded": 0, "b2_prob_skipped": 0, "no_trade": 0}
    total = 0.0
    for r in reserves:
        pnl, label = trade_pnl(b1, b2, r, avg_b2, penalty_mode, rng)
        total += pnl
        counts[label] += 1

    return SimResult(
        b1=b1, b2=b2, avg_b2=avg_b2,
        n_counterparties=n_counterparties,
        total_pnl=total,
        mean_pnl=total / n_counterparties,
        outcome_counts={k: v for k, v in counts.items() if v > 0},
    )


# ---------------------------------------------------------------------------
# Best-response search: pick (b1, b2) given assumed avg_b2
# ---------------------------------------------------------------------------

def best_response(
    avg_b2: float,
    penalty_mode: PenaltyMode = PenaltyMode.SUBTRACT,
    grid: Iterable[float] | None = None,
) -> tuple[float, float, float]:
    """Grid-search the (b1, b2) pair that maximises expected PnL per counterparty.

    Returns (best_b1, best_b2, best_expected_pnl).
    """
    if grid is None:
        grid = RESERVES  # search on the same 5-step grid as reserves

    best = (-np.inf, None, None)
    for b1 in grid:
        for b2 in grid:
            if b2 < b1:
                continue
            ev = expected_pnl_per_cp(b1, b2, avg_b2, penalty_mode)
            if ev > best[0]:
                best = (ev, b1, b2)
    ev, b1, b2 = best
    return float(b1), float(b2), float(ev)


# ---------------------------------------------------------------------------
# Fixed-point: avg_b2 such that everyone playing best-response produces it
# ---------------------------------------------------------------------------

def find_equilibrium_avg_b2(
    penalty_mode: PenaltyMode = PenaltyMode.SUBTRACT,
    tol: float = 0.5,
    max_iter: int = 50,
    start: float = 800.0,
    damping: float = 0.5,
) -> tuple[float, float, float]:
    """Iterate avg_b2 -> best_response_b2 until fixed point.

    Returns (avg_b2_eq, b1_eq, b2_eq).
    """
    avg = start
    for _ in range(max_iter):
        b1, b2, _ = best_response(avg, penalty_mode)
        new_avg = damping * avg + (1 - damping) * b2
        if abs(new_avg - avg) < tol:
            return new_avg, b1, b2
        avg = new_avg
    return avg, b1, b2


# ---------------------------------------------------------------------------
# Demo
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    print("=== Sanity check: single trades ===")
    for b1, b2, r in [(750, 850, 700), (750, 850, 800), (750, 850, 900)]:
        pnl, label = trade_pnl(b1, b2, r, avg_b2=830)
        print(f"  b1={b1} b2={b2} r={r}  ->  pnl={pnl:.2f}  ({label})")

    print("\n=== Expected PnL for a few strategies (assuming avg_b2 = 830) ===")
    for b1, b2 in [(700, 800), (750, 830), (770, 850), (790, 870), (800, 880)]:
        ev = expected_pnl_per_cp(b1, b2, avg_b2=830)
        print(f"  b1={b1} b2={b2}  ->  E[PnL/cp] = {ev:.3f}")

    print("\n=== Best response under SUBTRACT penalty, given avg_b2 = 830 ===")
    b1, b2, ev = best_response(avg_b2=830)
    print(f"  best (b1, b2) = ({b1}, {b2})   E[PnL/cp] = {ev:.3f}")

    print("\n=== Symmetric equilibrium (everyone best-responds to avg_b2) ===")
    avg_eq, b1_eq, b2_eq = find_equilibrium_avg_b2()
    print(f"  avg_b2* = {avg_eq:.2f}   (b1*, b2*) = ({b1_eq}, {b2_eq})")

    print("\n=== Monte-Carlo check at the equilibrium strategy ===")
    res = simulate(b1=b1_eq, b2=b2_eq, avg_b2=avg_eq, n_counterparties=200_000)
    print(res)

    print("\n=== Same, under PROBABILITY interpretation of the penalty ===")
    avg_eq2, b1_eq2, b2_eq2 = find_equilibrium_avg_b2(
        penalty_mode=PenaltyMode.PROBABILITY
    )
    print(f"  avg_b2* = {avg_eq2:.2f}   (b1*, b2*) = ({b1_eq2}, {b2_eq2})")
    res2 = simulate(b1_eq2, b2_eq2, avg_eq2, 200_000,
                    penalty_mode=PenaltyMode.PROBABILITY)
    print(res2)

=== Sanity check: single trades ===
  b1=750 b2=850 r=700  ->  pnl=170.00  (b1_hit)
  b1=750 b2=850 r=800  ->  pnl=70.00  (b2_clean)
  b1=750 b2=850 r=900  ->  pnl=0.00  (no_trade)

=== Expected PnL for a few strategies (assuming avg_b2 = 830) ===
  b1=700 b2=800  ->  E[PnL/cp] = 77.089
  b1=750 b2=830  ->  E[PnL/cp] = 84.588
  b1=770 b2=850  ->  E[PnL/cp] = 83.725
  b1=790 b2=870  ->  E[PnL/cp] = 79.412
  b1=800 b2=880  ->  E[PnL/cp] = 76.078

=== Best response under SUBTRACT penalty, given avg_b2 = 830 ===
  best (b1, b2) = (750.0, 835.0)   E[PnL/cp] = 85.000

=== Symmetric equilibrium (everyone best-responds to avg_b2) ===
  avg_b2* = 834.73   (b1*, b2*) = (750.0, 835.0)

=== Monte-Carlo check at the equilibrium strategy ===
b1=750.0  b2=835.0  avg_b2=834.7  N=200000
  total PnL = 16979600.00   mean PnL/cp = 84.8980
  outcomes  = {'b1_hit': 66445, 'b2_clean': 66870, 'no_trade': 66685}

=== Same, under PROBABILITY interpretation of the penalty ===
  avg_b2* = 834.73   (b1*, b2*) = (7